# FraudShield Model Training Notebook
**Pipeline:** Data Loading → EDA → Feature Engineering → Split → Pipeline → Evaluation → Export

**Selected Features (hasil feature selection):**
- `amt`, `hour`, `month`, `amt_log`, `amt_mean_per_card`, `amt_ratio`
- `txn_count_per_card`, `unique_merchants`, `unique_categories`, `amt_std_per_card`
- `category_food_dining`, `category_gas_transport`, `category_grocery_pos`, `category_kids_pets`
- `category_misc_net`, `category_misc_pos`, `category_personal_care`, `category_shopping_net`
- `merchant`, `city`

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
from pathlib import Path

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, average_precision_score
)

# Imbalanced-learn
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline



## 2. Load Dataset

In [ ]:
# Ganti path sesuai lokasi file kamu
DATA_PATH = '../data/raw/fraud_test.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
print(f'Fraud rate   : {df["is_fraud"].mean():.4f} ({df["is_fraud"].sum()} kasus fraud)')
df.head()

## 3. EDA (Exploratory Data Analysis)

In [ ]:

print(df.shape)


print(df.dtypes)


print(df.isnull().sum()[df.isnull().sum() > 0])


print(df['is_fraud'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribusi amount
axes[0].hist(df[df['is_fraud']==0]['amt'], bins=50, alpha=0.6, label='Legit', color='steelblue')
axes[0].hist(df[df['is_fraud']==1]['amt'], bins=50, alpha=0.6, label='Fraud', color='red')
axes[0].set_xlabel('Amount (USD)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribusi Amount per Kelas')
axes[0].legend()
axes[0].set_xlim(0, 2000)

# Target proportion
df['is_fraud'].value_counts().plot.pie(
    ax=axes[1], autopct='%1.1f%%',
    labels=['Legitimate','Fraud'], colors=['steelblue','red']
)
axes[1].set_title('Proporsi Kelas Target')

plt.tight_layout()
plt.show()

## 4. Data Cleaning & Type Conversion

In [ ]:
# Parse datetime
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'], infer_datetime_format=True)
df['dob'] = pd.to_datetime(df['dob'], infer_datetime_format=True)

# Gabungkan first + last name
df['full_name'] = df['first'] + ' ' + df['last']
df.drop(['first', 'last'], axis=1, inplace=True)

print('Datetime columns parsed')
df.dtypes

## 5. Feature Engineering

In [ ]:
# Temporal features
df['hour']        = df['trans_date_trans_time'].dt.hour
df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
df['month']       = df['trans_date_trans_time'].dt.month
df['is_night']    = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)

# Age features 
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
df['age_group'] = pd.cut(
    df['age'], bins=[0, 25, 40, 60, 100],
    labels=['<25', '25-40', '40-60', '>60']
)

# Amount features 
df['amt_log']         = np.log1p(df['amt'])
df['is_round_amount'] = ((df['amt'] % 10 == 0) & (df['amt'] > 0)).astype(int)

# Geographic features 
from math import radians, sin, cos, asin, sqrt

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * asin(sqrt(a))

df['distance_to_merch'] = df.apply(
    lambda r: haversine(r['lat'], r['long'], r['merch_lat'], r['merch_long']), axis=1
)
df['is_far_from_home'] = (df['distance_to_merch'] > 50).astype(int)

# Population & gender encoding 
df['city_pop_log'] = np.log1p(df['city_pop'])
df['gender_enc']   = df['gender'].map({'M': 1, 'F': 0}).fillna(0).astype(int)

# Drop kolom tidak diperlukan 
DROP_COLS = ['street', 'trans_num', 'unix_time', 'trans_date_trans_time', 'dob', 'full_name']
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

print(f'Shape setelah feature engineering: {df.shape}')
df.head()

## 6. Split Data (Train / Val / Test)

In [ ]:
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

# 60% train | 20% val | 20% test — stratified
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f'Train : {X_train.shape} | fraud rate: {y_train.mean():.4f}')
print(f'Val   : {X_val.shape}   | fraud rate: {y_val.mean():.4f}')
print(f'Test  : {X_test.shape}  | fraud rate: {y_test.mean():.4f}')

In [ ]:
# Card-level aggregation
agg_train = X_train.groupby('cc_num').agg(
    amt_mean_per_card  = ('amt', 'mean'),
    amt_std_per_card   = ('amt', 'std'),
    txn_count_per_card = ('amt', 'count'),
    unique_merchants   = ('merchant', 'nunique'),
    unique_categories  = ('category', 'nunique')
).reset_index()

# Merge ke semua split
for split_X in [X_train, X_val, X_test]:
    split_X.merge(agg_train, on='cc_num', how='left', suffixes=('', '_agg'))

X_train = X_train.merge(agg_train, on='cc_num', how='left')
X_val   = X_val.merge(agg_train, on='cc_num', how='left')
X_test  = X_test.merge(agg_train, on='cc_num', how='left')

# Fill NaN (kartu baru) dengan median dari train
AGG_COLS = ['amt_mean_per_card', 'amt_std_per_card', 'txn_count_per_card',
            'unique_merchants', 'unique_categories']
medians = {col: X_train[col].median() for col in AGG_COLS}

for col, med in medians.items():
    X_train[col] = X_train[col].fillna(med)
    X_val[col]   = X_val[col].fillna(med)
    X_test[col]  = X_test[col].fillna(med)

# amt_ratio hitung setelah merge
for split_X in [X_train, X_val, X_test]:
    split_X['amt_ratio'] = split_X['amt'] / split_X['amt_mean_per_card'].replace(0, 1)

# Drop cc_num 
for split_X in [X_train, X_val, X_test]:
    if 'cc_num' in split_X.columns:
        split_X.drop('cc_num', axis=1, inplace=True)

print('Card-level aggregation selesai')
print(f'X_train shape: {X_train.shape}')

## 7. Build Pipeline

In [ ]:
# Custom Frequency Encoder
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Encode categorical sebagai frekuensi kemunculan di training set."""

    def __init__(self):
        self.maps = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X
        for col in X.columns:
            self.maps[col] = X[col].value_counts().to_dict()
        return self

    def transform(self, X):
        X = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()
        for col in X.columns:
            # Unknown → 0 (bukan NaN agar imputer tidak perlu)
            X[col] = X[col].map(self.maps[col]).fillna(0)
        return X.values


In [ ]:
# Kolom per tipe 
numeric_cols = [
    'amt', 'zip', 'lat', 'long', 'city_pop',
    'merch_lat', 'merch_long', 'hour', 'day_of_week', 'month',
    'distance_to_merch', 'amt_log', 'is_round_amount',
    'amt_mean_per_card', 'amt_ratio', 'txn_count_per_card',
    'unique_merchants', 'unique_categories', 'amt_std_per_card',
    'age', 'city_pop_log', 'gender_enc', 'is_night', 'is_weekend',
    'is_far_from_home'
]

onehot_cols = ['category']
freq_cols   = ['merchant', 'city', 'job', 'state']

# Filter: pastikan kolom ada di X_train
numeric_cols = [c for c in numeric_cols if c in X_train.columns]
onehot_cols  = [c for c in onehot_cols  if c in X_train.columns]
freq_cols    = [c for c in freq_cols    if c in X_train.columns]

print(f'Numeric : {len(numeric_cols)} kolom')
print(f'OneHot  : {onehot_cols}')
print(f'Freq    : {freq_cols}')

# Sub-pipelines 
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

onehot_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

freq_pipe = Pipeline([
    ('imputer',      SimpleImputer(strategy='most_frequent')),
    ('freq_encoder', FrequencyEncoder()),
    ('scaler',       StandardScaler()),   
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num',    num_pipe,    numeric_cols),
        ('onehot', onehot_pipe, onehot_cols),
        ('freq',   freq_pipe,   freq_cols),
    ],
    remainder='drop',
)

# Full Pipeline 
full_pipeline = ImbPipeline([
    ('preprocessor',  preprocessor),
    ('smote_tomek',   SMOTETomek(random_state=42)),
    ('selector',      SelectFromModel(
                          RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
                          threshold='median'
                      )),
    ('model',         RandomForestClassifier(
                          n_estimators=200,
                          max_depth=15,
                          min_samples_leaf=5,
                          class_weight='balanced_subsample',
                          random_state=42,
                          n_jobs=-1,
                      )),
])



## 8. Training

In [ ]:
t0 = datetime.now()
full_pipeline.fit(X_train, y_train)
elapsed = (datetime.now() - t0).seconds
print(f'Training selesai dalam {elapsed} detik')

## 9. Evaluasi

In [ ]:
y_pred = full_pipeline.predict(X_test)
y_prob = full_pipeline.predict_proba(X_test)[:, 1]

print('=== Classification Report (Test) ===')
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

# AUC
auc_train = roc_auc_score(y_train, full_pipeline.predict_proba(X_train)[:, 1])
auc_val   = roc_auc_score(y_val,   full_pipeline.predict_proba(X_val)[:, 1])
auc_test  = roc_auc_score(y_test,  y_prob)

print(f'\nAUC-ROC Train : {auc_train:.4f}')
print(f'AUC-ROC Val   : {auc_val:.4f}')
print(f'AUC-ROC Test  : {auc_test:.4f}')

ap_score = average_precision_score(y_test, y_prob)
print(f'Average Precision: {ap_score:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legit','Fraud'], yticklabels=['Legit','Fraud'])
axes[0].set_title('Confusion Matrix (Test Set)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Risk Score Distribution
axes[1].hist(y_prob[y_test==0], bins=50, alpha=0.6, label='Legit', color='steelblue')
axes[1].hist(y_prob[y_test==1], bins=50, alpha=0.6, label='Fraud', color='red')
axes[1].axvline(0.75, color='red', linestyle='--', label='Fraud threshold (0.75)')
axes[1].axvline(0.45, color='orange', linestyle='--', label='Review threshold (0.45)')
axes[1].set_xlabel('Risk Score (Predicted Probability)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribusi Risk Score')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Feature Selection — Fitur yang Terpilih

In [ ]:
# Ambil nama fitur dari setiap transformer
num_names    = numeric_cols
onehot_names = (full_pipeline.named_steps['preprocessor']
                    .named_transformers_['onehot']
                    .named_steps['encoder']
                    .get_feature_names_out(onehot_cols)
                    .tolist())
freq_names   = freq_cols

all_feature_names = np.array(num_names + onehot_names + freq_names)

selector = full_pipeline.named_steps['selector']
selected_mask = selector.get_support()
selected_features = all_feature_names[selected_mask].tolist()

print(f'Fitur sebelum seleksi : {len(all_feature_names)}')
print(f'Fitur setelah seleksi : {len(selected_features)}')
print('\nFitur yang dipilih:')
for f in selected_features:
    print(f'  - {f}')

## 11. Export Model

In [ ]:
OUTPUT_DIR = Path('../backend/trained_models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Simpan pipeline
model_path = OUTPUT_DIR / 'fraud_pipeline.pkl'
joblib.dump(full_pipeline, model_path)
print(f'Model saved → {model_path}')

# Simpan metadata
metadata = {
    'model_name': 'FraudShield Fraud Detection Pipeline',
    'version': '1.0.0',
    'trained_at': datetime.now().isoformat(),
    'algorithm': 'RandomForestClassifier',
    'pipeline_steps': list(full_pipeline.named_steps.keys()),
    'selected_features': selected_features,
    'all_features_before_selection': all_feature_names.tolist(),
    'metrics': {
        'auc_roc_train': round(auc_train, 4),
        'auc_roc_val':   round(auc_val, 4),
        'auc_roc_test':  round(auc_test, 4),
        'average_precision': round(ap_score, 4),
    },
    'thresholds': {'fraud': 0.75, 'review': 0.45},
    'training_config': {
        'train_size': 0.6, 'val_size': 0.2, 'test_size': 0.2,
        'stratified': True, 'random_state': 42,
    },
    'card_agg_medians': medians,
}

with open(OUTPUT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)


print(f'\nSelected features ({len(selected_features)}):')
for f in selected_features:
    print(f'  - {f}')